In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.fetcher import OptionChainFetcher
from data.cleaner import clean_option_chain
from models.surface import IVSurface
from models.density import (
    breeden_litzenberger_density,
    compute_moments,
    compute_tail_mass
)
from analytics.tail_risk import (
    compute_tail_metrics,
    compute_distribution_moments,
    generate_tail_risk_report
)

In [ ]:
# Fetch and process data
fetcher = OptionChainFetcher('SPY')
chain = fetcher.fetch_current_chain()
clean_chain = clean_option_chain(chain)
surface = IVSurface(clean_chain)

In [ ]:
# Extract risk-neutral density for 30-day expiry
strikes, density = breeden_litzenberger_density(surface, expiry=30)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=strikes,
    y=density,
    mode='lines',
    fill='tozeroy',
    name='Risk-Neutral Density'
))
fig.update_layout(
    title='Risk-Neutral Density (30 Days)',
    xaxis_title='Strike (Moneyness)',
    yaxis_title='Probability Density'
)
fig.show()

In [ ]:
# Compute distribution moments
moments = compute_moments(strikes, density)

print("Distribution Moments:")
print(f"  Mean: {moments['mean']:.4f}")
print(f"  Variance: {moments['variance']:.6f}")
print(f"  Skewness: {moments['skewness']:.4f}")
print(f"  Kurtosis: {moments['kurtosis']:.4f}")

In [ ]:
# Compute tail mass
left_tail = compute_tail_mass(strikes, density, threshold=0.9, tail='left')
right_tail = compute_tail_mass(strikes, density, threshold=1.1, tail='right')

print(f"\nTail Mass:")
print(f"  Left tail (<90%): {left_tail*100:.2f}%")
print(f"  Right tail (>110%): {right_tail*100:.2f}%")
print(f"  Tail asymmetry: {left_tail/max(right_tail, 0.001):.2f}x")

In [ ]:
# Compare densities across expiries
fig = go.Figure()

for exp in [14, 30, 60, 90]:
    try:
        strikes_exp, density_exp = breeden_litzenberger_density(surface, expiry=exp)
        fig.add_trace(go.Scatter(
            x=strikes_exp,
            y=density_exp,
            mode='lines',
            name=f'{exp}D'
        ))
    except Exception as e:
        print(f"Could not compute density for {exp}D: {e}")

fig.update_layout(
    title='Risk-Neutral Density Term Structure',
    xaxis_title='Strike (Moneyness)',
    yaxis_title='Probability Density'
)
fig.show()

In [ ]:
# Generate tail risk report
report = generate_tail_risk_report(surface, expiry=30)

print("\nTail Risk Report:")
for key, value in report.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")